# Change Detection

#### Den här notebooken utför **image differencing** baserat på preprocessade indexraster från `preprocessing.ipynb`

### Importer
Importerar bibliotek (som `numpy`, `xarray` och `geopandas`) samt funktioner från `config.py` som krävs för att köra koden.

In [ ]:
import os
import numpy as np
import pandas as pd
import rioxarray as rxr
import geopandas as gpd
import matplotlib.pyplot as plt

from config import (
    build_analysis_output_dirs,
    build_analysis_satellites,
    build_stack,
    load_and_prepare_scene_logs,
    print_satellite_setup,
    select_best_scenes,
    OPEN_WETLAND_AREA,
    select_all_clear_scenes,
    select_scenes_hybrid
    )

### Konfiguration och initiering
Välj alt studieområde (`area`) och satellitsensor (`sensor`). I detta steg skapas även de filvägar (mappar) dit resultaten kommer att sparas.

In [ ]:
# ============================================
# Konfigurera sökvägar och studieområde
# ============================================
area = "LF"  # OBS: här väljs studieområdet
sensor = "sentinel2"  # "landsat" eller "sentinel2"

satellites = build_analysis_satellites(area, sensor=sensor)

analysis_output_dirs = build_analysis_output_dirs(sensor=sensor)
output_dir_change_images = analysis_output_dirs["change_images"]
os.makedirs(output_dir_change_images, exist_ok=True)

print_satellite_setup(area, satellites, sensor=sensor)

### Steg 1: Inläsning av scenloggar
Skriptet hämtar metadata om de tillgängliga satellitbilderna för området från CSV-loggar. 
Scenloggarna är en del av output från `preprocessing.ipynb`

In [ ]:
# =============================================
# Steg 1: Läs in och kombinera loggar
# =============================================
log = load_and_prepare_scene_logs(satellites, area)
log = log.sort_values("date").reset_index(drop=True)

### Steg 1.5: Frivillig filtrering av Landsat 7
Eftersom Landsat 7 (från maj 2003) i regel har dataförlust (SLC-off ränder) finns här möjligheten att exkludera alla sådana bilder. Endast bilder från exempelvis L05, L08 och L09 kommer då sparas vidare. S2 påverkas inte.

In [ ]:
# Valbart inför körning: sätt True för att exkludera enbart Landsat 7
exclude_l07_this_run = True

if exclude_l07_this_run:
    before_n = len(log)
    if "sat_name" in log.columns:
        # Strikt filter: ta bara bort L07-rader, behåll L05/L0809
        log = log[log["sat_name"].astype(str).str.upper() != "L07"].copy()
    elif "scene_name" in log.columns:
        # Fallback om sat_name saknas i loggen
        log = log[~log["scene_name"].astype(str).str.contains("LE07", case=False, na=False)].copy()
    print(f"L07 exkluderat denna körning: {before_n - len(log)} scener borttagna")


### Steg 2: Urval av molnfria scener
Genom att filtrera listan  bibehålls enbart molnfria scener. En referens med de bilder som har godkänds för analysen sparas sedan till disk som en upplysning.

In [ ]:
# =============================================
# Steg 2: Välj molnfria scener (alla eller en per år)
# =============================================

# Använd för Sentinel-2
clear_scenes = select_all_clear_scenes(log) # Väljer alla molnfria scener
print(f"Antal valda molnfria scener: {len(clear_scenes)}")
print(clear_scenes[["year", "date", "days_from_july15", "sat_name", "cloud_pct"]])

# Använd för Landsat
# best_scenes = select_best_scenes(log) # Väljer en bästa molnfri scen per år närmast 15 juli
# print(f"Antal valda bästa scener: {len(best_scenes)}")
# print(best_scenes[["year", "date", "days_from_july15", "sat_name", "cloud_pct"]])

# Kör denna för Landsat om du vill skapa en baseline baserat på flera år
# Väljer alla molnfira bilder 1984-1989 för att skapa baseline, väljer bästa molfria scen för enskilda år. 
# scenes = select_scenes_hybrid(log, period_start=1984, period_end=1989)
# print(f"Antal använd scener: {len(scenes)}")
# print(scenes[["year", "date", "days_from_july15", "sat_name", "cloud_pct"]])

# Spara som CSV
os.makedirs(output_dir_change_images, exist_ok=True)
out_path = os.path.join(output_dir_change_images, f"{area}_{sensor}_selected_scenes.csv")
clear_scenes.to_csv(out_path, index=False)
print(f"Sparad: {out_path}")

### Steg 3: Bygga Datakuber (xarray-stackar) per Index
Skriptet läser in index-rastren (till exempel EVI eller MNDWI) och stackar ihop dem över tid i formatet av en datacube (`stack`). Den rumsliga representationen tillåter beräkningar rakt igenom tidsseriens pixelvärden.

In [ ]:
# =============================================
# Steg 3: Bygg xarray-stack per index CLEAR scenes
# =============================================

# Använd för Sentinel-2, skapar stacks från alla molnfria scener. 
stacks = {}
for index_name in ["NDVI", "MNDWI", "NDMI", "NDWI", "EVI"]:
    print(f"Bygger stack för {index_name}...")
    stacks[index_name] = build_stack(clear_scenes, index_name, area)
    print(f"  Shape: {stacks[index_name].shape}")


# Använd för Landsat, skapar stacks från bästa scen per år.
# stacks = {}
# for index_name in ["MNDWI", "EVI"]:
#     print(f"Bygger stack för {index_name}...")
#     stacks[index_name] = build_stack(best_scenes, index_name, area)
#     print(f"  Shape: {stacks[index_name].shape}")


# Använd för Landsat. Stacks för hybridvalda scener (alla molnfria 1984-1989 + bästa scen per år efter det)
# stacks = {}
# for index_name in ["NDVI", "MNDWI", "NDMI", "NDWI", "EVI"]:
#     print(f"Bygger stack för {index_name}...")
#     stacks[index_name] = build_stack(scenes, index_name, area)
#     print(f"  Shape: {stacks[index_name].shape}")

### Steg 4: Bildförändring ("Change Image") mellan två tidsintervall
Ett tidigt år (`year_old`) ställs här mot ett senare år (`year_recent`). Skillnaden i pixelvärden beräknas mellan åren genom att subtrahera den senare bilden mot den tidigare. Resultatet visualizeras därefter i figurerna som en förändringskarta, och kan klippas med en shapefil, innan slutresultaten lagras ner (i GeoTIFF).

In [ ]:
# =============================================
# Steg 4: Change image mellan två år
# =============================================
year_old    = 2021
year_recent = 2025

# Sätt clip_shp till sökväg för att klippa till ett delområde, eller None för hela studieområdet.
clip_shp = None # Exempel: OPEN_WETLAND_AREA["SHAPE_FILE"]
if area == "TAM":
    clip_shp = r"D:\Masteruppsats\Studieomraden\Omr_polygon\TAM_lake.shp" # Vatten-polygon TAM-området

cmaps = {"NDVI": "RdYlGn", "MNDWI": "RdBu", "NDMI": "BrBG", "NDWI": "RdYlBu", "EVI": "PiYG"}

if clip_shp is not None:
    aoi = gpd.read_file(clip_shp)
    aoi = aoi.to_crs(next(iter(stacks.values())).rio.crs)

for index_name, stack in stacks.items():
    available_years = stack.time.dt.year.values

    idx_old    = np.flatnonzero(available_years == year_old)
    idx_recent = np.flatnonzero(available_years == year_recent)

    if len(idx_old) == 0:
        print(f"[{index_name}] Inget år {year_old} finns i stacken. Tillgängliga år: {sorted(set(available_years))}")
        continue
    if len(idx_recent) == 0:
        print(f"[{index_name}] Inget år {year_recent} finns i stacken. Tillgängliga år: {sorted(set(available_years))}")
        continue

    # Gäller endast Sentinel-2. Medelvärde över alla scener per år (ändrad 2026-03-31).
    img_old    = stack.isel(time=idx_old).mean(dim="time")
    img_recent = stack.isel(time=idx_recent).mean(dim="time")


    # Klipp bilderna om clip_shp är definierad
    if clip_shp is not None:
        img_old    = img_old.rio.clip(aoi.geometry, aoi.crs, invert=True)
        img_recent = img_recent.rio.clip(aoi.geometry, aoi.crs, invert=True)

    # Beräkning av change image
    actual_year_old    = year_old
    actual_year_recent = year_recent

    change = img_recent - img_old

    # Anpassa figurstorlek efter rasterförhållandet för mer realistiska proportioner
    ny, nx = change.shape
    panel_h = 6
    panel_w = panel_h * (nx / ny)
    fig, ax = plt.subplots(figsize=(panel_w, panel_h))
    change.plot(ax=ax, cmap=cmaps.get(index_name, "RdYlGn"), center=0, add_colorbar=True)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title(f"{index_name}-förändring {actual_year_old}–{actual_year_recent} ({area})")
    plt.tight_layout()
    plt.show()

    # Spara som GeoTIFF
    change_out = change.rio.write_crs(stack.rio.crs)
    change_out = change_out.rio.write_nodata(np.nan)
    out_path = os.path.join(output_dir_change_images, f"{area}_{index_name}_change_{actual_year_old}_{actual_year_recent}.tif")
    change_out.rio.to_raster(out_path, overwrite=True)
    print(f"Sparad: {out_path}")

    # Listar vilka scener som ingår change image varje år
    dates_old = [str(np.datetime_as_string(t, unit="D")).replace("-", "") for t in stack.time.values[idx_old]]
    dates_recent = [str(np.datetime_as_string(t, unit="D")).replace("-", "") for t in stack.time.values[idx_recent]]
    print(f"  Antal bilder {year_old}: {len(idx_old)} ({', '.join(dates_old)})")
    print(f"  Antal bilder {year_recent}: {len(idx_recent)} ({', '.join(dates_recent)})")

### Steg 5a: Beräkna avvikelse från den historiska Baslinjen (Baseline)
Gäller endast Landsat data. I det här steget skapas en change image från en vald referensperiod (t.ex. 1984–1989). Senare bilder subtraheras från referensperioden vilket visar avvikelsen från referensperioden. Skapar ett raster med baseline-prioden samt change image. Alla raster exporteras i GeoTIFF-format.

In [ ]:
# =============================================
# Steg 5: Baseline och avvikelse från enskilda år
# =============================================
baseline_start = "1984"
baseline_end   = "1989"
compare_years  = [2024, 2025]

cmaps = {"NDVI": "RdYlGn", "MNDWI": "RdBu", "NDMI": "BrBG", "NDWI": "RdYlBu", "EVI": "PiYG"}

# Sätt clip_shp till sökväg för att klippa till ett delområde, eller None för hela studieområdet.
clip_shp = None #Exempel: OPEN_WETLAND_AREA["SHAPE_FILE"] för att klippa till den öppna våtmarken i GRM
if area == "TAM":
    clip_shp = r"D:\Masteruppsats\Studieomraden\Omr_polygon\TAM_lake.shp" # Vatten-polygon TAM-området

if clip_shp is not None:
    aoi = gpd.read_file(clip_shp)
    aoi = aoi.to_crs(next(iter(stacks.values())).rio.crs)

for index_name, stack in stacks.items():
    baseline = stack.sel(time=slice(baseline_start, baseline_end)).mean(dim="time")

    # Klipp baseline om clip_shp är definierad
    if clip_shp is not None:
        baseline = baseline.rio.clip(aoi.geometry, aoi.crs, invert=True)

    n_baseline = len(stack.sel(time=slice(baseline_start, baseline_end)).time)
    print(f"{index_name}: {n_baseline} scener i baseline ({baseline_start}–{baseline_end})")

    # Spara baseline
    base_out = baseline.rio.write_crs(stack.rio.crs)
    base_out = base_out.rio.write_nodata(np.nan)

    out_path = os.path.join(output_dir_change_images, f"{area}_{index_name}_baseline_yr_{baseline_start}_{baseline_end}.tif")
    if os.path.exists(out_path):
        os.remove(out_path)
    base_out.rio.to_raster(out_path)
    print(f"Sparad: {out_path}")

    for year in compare_years:
        matches = np.flatnonzero(stack.time.dt.year.values == year)
        if len(matches) == 0:
            print(f"  [{index_name}] Inget år {year} i stacken.")
            continue

        # Innan jag ändrade till att ta medelvärde över alla scener per år (ändrad 2026-03-31)
        # img = stack.isel(time=matches[0])
        # actual_year = int(img.time.dt.year.values)
        img = stack.isel(time=matches).mean(dim="time")
        dates_used = stack.isel(time=matches).time.values
        actual_year = year

        n_img = len(stack.isel(time=matches).time)
        print(f"{index_name}: {n_img} scener för år {year} ({', '.join([str(np.datetime_as_string(t, unit='D')).replace('-', '') for t in dates_used])})")
        
        img_out = img.rio.write_crs(stack.rio.crs)
        img_out = img_out.rio.write_nodata(np.nan)
        out_path = os.path.join(output_dir_change_images, f"{area}_{index_name}_compare_yr_{year}.tif")
        img_out.rio.to_raster(out_path, overwrite=True)
        print(f"Sparad: {out_path}")

        # Klipp bilden om clip_shp är definierad
        if clip_shp is not None:
            img = img.rio.clip(aoi.geometry, aoi.crs, invert=True)

        print(f"  Använder scen från {actual_year} för jämförelseår {year}")

        deviation = img - baseline

        vmin = float(np.nanmin([baseline.values, img.values]))
        vmax = float(np.nanmax([baseline.values, img.values]))
        abs_max = float(np.nanmax(np.abs(deviation.values)))

        # Anpassa figurstorlek efter rasterförhållandet för mer realistiska proportioner
        ny, nx = baseline.shape
        panel_h = 5
        panel_w = panel_h * (nx / ny)
        fig, axes = plt.subplots(1, 3, figsize=(3 * panel_w, panel_h))
        baseline.plot(ax=axes[0], cmap=cmaps.get(index_name, "RdYlGn"), vmin=vmin, vmax=vmax, add_colorbar=True)
        axes[0].set_aspect("equal")
        axes[0].set_xticks([])
        axes[0].set_yticks([])
        axes[0].set_xlabel("")
        axes[0].set_ylabel("")
        axes[0].set_title(f"Baseline {baseline_start}–{baseline_end}")

        img.plot(ax=axes[1], cmap=cmaps.get(index_name, "RdYlGn"), vmin=vmin, vmax=vmax, add_colorbar=True)
        axes[1].set_aspect("equal")
        axes[1].set_xticks([])
        axes[1].set_yticks([])
        axes[1].set_xlabel("")
        axes[1].set_ylabel("")
        axes[1].set_title(f"År {actual_year}")

        deviation.plot(ax=axes[2], cmap=cmaps.get(index_name, "RdYlGn"), center=0, vmin=-abs_max, vmax=abs_max, add_colorbar=True)
        axes[2].set_aspect("equal")
        axes[2].set_xticks([])
        axes[2].set_yticks([])
        axes[2].set_xlabel("")
        axes[2].set_ylabel("")
        axes[2].set_title("Avvikelse från baseline")

        fig.suptitle(f"{index_name} – {area}")
        plt.tight_layout()
        plt.show()

        # Spara avvikelse
        dev_out = deviation.rio.write_crs(stack.rio.crs)
        dev_out = dev_out.rio.write_nodata(np.nan)
        out_path = os.path.join(output_dir_change_images, f"{area}_{index_name}_deviation_baseline_{actual_year}.tif")
        dev_out.rio.to_raster(out_path, overwrite=True)
        print(f"Sparad: {out_path}")

### Steg 5b: Statistisk Baseline-sammanställning per zon
Deskriptiv statistik (medelvärden och standardavvikelse) räknas ut enbart för det område som befinner sig i avsedd shapemask (t.ex. en "open wetland"-poylgon). Resultaten lagras sedan som en CSV-tabell.

In [ ]:
# =============================================
# Steg 5b: Statistik inom open wetland
# =============================================
stat_shp = OPEN_WETLAND_AREA[f"{area}_wetland"]
rows = []

for index_name in ["NDVI", "MNDWI", "NDMI", "NDWI", "EVI"]:
    # Läs in baseline
    baseline_path = os.path.join(output_dir_change_images, f"{area}_{index_name}_baseline_yr_{baseline_start}_{baseline_end}.tif")
    if not os.path.exists(baseline_path):
        print(f"Saknas: {baseline_path}")
        continue
    baseline_rast    = rxr.open_rasterio(baseline_path, masked=True).squeeze()
    aoi_stat         = gpd.read_file(stat_shp).to_crs(baseline_rast.rio.crs)
    baseline_clipped = baseline_rast.rio.clip(aoi_stat.geometry, aoi_stat.crs)

    for year in compare_years:
        # Läs in jämförelseår
        img_path = os.path.join(output_dir_change_images, f"{area}_{index_name}_compare_yr_{year}.tif")
        if not os.path.exists(img_path):
            print(f"Saknas: {img_path}")
            continue
        img_rast    = rxr.open_rasterio(img_path, masked=True).squeeze()
        img_clipped = img_rast.rio.clip(aoi_stat.geometry, aoi_stat.crs)

        # Läs in avvikelse
        dev_path = os.path.join(output_dir_change_images, f"{area}_{index_name}_deviation_baseline_{year}.tif")
        if not os.path.exists(dev_path):
            print(f"Saknas: {dev_path}")
            continue
        dev_rast    = rxr.open_rasterio(dev_path, masked=True).squeeze()
        dev_clipped = dev_rast.rio.clip(aoi_stat.geometry, aoi_stat.crs)

        print(f"  --- {index_name} {year} inom open wetland ---")
        print(f"  Baseline mean ± std:  {float(baseline_clipped.mean()):.4f} ± {float(baseline_clipped.std()):.4f}")
        print(f"  {year} mean ± std:    {float(img_clipped.mean()):.4f} ± {float(img_clipped.std()):.4f}")
        print(f"  Avvikelse mean ± std: {float(dev_clipped.mean()):.4f} ± {float(dev_clipped.std()):.4f}")

        rows.append({
            "area":           area,
            "index":          index_name,
            "baseline":       f"{baseline_start}-{baseline_end}",
            "compare_year":   year,
            "baseline_mean":  round(float(baseline_clipped.mean()), 4),
            "baseline_std":   round(float(baseline_clipped.std()),  4),
            "year_mean":      round(float(img_clipped.mean()),      4),
            "year_std":       round(float(img_clipped.std()),       4),
            "deviation_mean": round(float(dev_clipped.mean()),      4),
            "deviation_std":  round(float(dev_clipped.std()),       4),
        })

df_stats = pd.DataFrame(rows)
out_path_csv = os.path.join(output_dir_change_images, f"statistics_{sensor}_{area}_baseline_{compare_years[0]}.csv")
df_stats.to_csv(out_path_csv, index=False)
print(f"Sparad: {out_path_csv}")